# 09 — RQ1, baseline phẳng 4 lớp (tham chiếu)

**Vì sao cần baseline**

Kiến trúc chính của đề tài là cascade hai tầng: tầng 1 lọc review actionable, tầng 2 phân BUG, FEATURE, UX. Để biết cách chia hai tầng
có đáng dùng hay không, cần một mốc so sánh: một mô hình phân thẳng 4 lớp BUG, FEATURE, UX, OTHER trong một lần.

**Vai trò theo protocol**

Baseline là ViSoBERT và chỉ dùng để đối chiếu, không đưa vào các bước sau (gom cụm, sinh requirement).

**Dữ liệu huấn luyện**

Toàn bộ 1.200 review Train với nhãn 4 lớp, giữ cả review ambiguous vì mỗi review vẫn có một nhãn gold. OTHER chiếm khoảng 72% Train nên
dùng weighted cross-entropy như các mô hình khác.

**Quy trình tune**

Giống tầng 1: dò learning rate {1e-5, 2e-5, 3e-5} và số epoch {3, 4, 5} với một seed, chọn theo Macro-F1 4 lớp trên Validation, rồi
chạy lại cấu hình thắng với ba seed. Mô hình của ba seed được lưu lại để Giai đoạn 10 so cascade với baseline trên Hold-out.

**So sơ bộ với cascade**

Phần cuối notebook so baseline với cascade ViSoBERT trên Validation, bằng cùng thước đo Macro-F1 4 lớp.

**Vì sao nạp lại checkpoint tầng 2**

Để ghép cascade trên toàn bộ Validation, cần nhãn tầng 2 cho mọi review mà tầng 1 gọi là actionable, kể cả các review OTHER bị gọi
nhầm, trong khi Giai đoạn 7 chỉ dự đoán tầng 2 trên phần gold-actionable. Vì vậy mô hình tầng 2 đã lưu ở Giai đoạn 7 được nạp lại và dự
đoán trên cả 350 review. Notebook cần gắn output của notebook 07 (thư mục `checkpoints/`) làm input.

**Chỉ để tham khảo**

Toàn bộ so sánh ở đây chỉ trên Validation; kết luận cascade hay baseline tốt hơn để ở Giai đoạn 10 trên Hold-out.

**Lấy mã nguồn**

Giống các notebook trước, mã nguồn được lấy từ repository riêng tư bằng token trong Kaggle Secrets, và token được gỡ khỏi cấu hình git
ngay sau khi clone.

In [ ]:
import os, subprocess, sys, json, gc
from kaggle_secrets import UserSecretsClient
REPO_URL = 'github.com/dnhphuong-05/kltn-vn-app-review-requirements.git'
REPO_DIR = '/kaggle/working/repo'
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{REPO_URL}', REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone thất bại: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
sys.path.append(os.path.join(REPO_DIR, '02_src'))
import yaml, transformers, sentencepiece
from src.utils.seeds import set_seed
set_seed(42)
CFG = yaml.safe_load(open(os.path.join(REPO_DIR, '03_configs', 'frozen', 'classification.yaml'), encoding='utf-8'))
print('transformers', transformers.__version__, '| sentencepiece', sentencepiece.__version__)
print('commit:', subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())

## Dữ liệu 4 lớp

In [ ]:
import pandas as pd, numpy as np, torch
from src.classification.flat_baseline.mapping import to_flat, FLAT_LABELS

def find_data_dir(root='/kaggle/input', target='shopee_train.csv'):
    for dirpath, _, files in os.walk(root):
        if target in files:
            return dirpath
    found = [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs]
    raise FileNotFoundError(f'Không tìm thấy {target} trong {root}. Các file hiện có:\n' + ('\n'.join(found) or '(trống)'))

DATA_DIR = find_data_dir()
train_df = pd.read_csv(os.path.join(DATA_DIR, 'shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(os.path.join(DATA_DIR, 'shopee_validation.csv'), encoding='utf-8-sig')
assert not set(train_df['review_id']) & set(val_df['review_id']), 'Train và Validation trùng review'
FLAT_TRAIN, FLAT_VAL = to_flat(train_df), to_flat(val_df)
print('Train:', FLAT_TRAIN['gold_label'].value_counts().reindex(FLAT_LABELS).to_dict())
print('Validation:', FLAT_VAL['gold_label'].value_counts().reindex(FLAT_LABELS).to_dict())
print('Các file trên Kaggle (thư mục dữ liệu):', sorted(os.listdir(DATA_DIR)))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có')
from src.utils.kaggle_paths import working_dir
OUT = working_dir('rq1_flat')
CKPT = os.path.join(OUT, 'checkpoints')
SEEDS = CFG['learning_curve_seeds']

## Baseline: dò hyperparameter

**Lưới**

Train 4 lớp có cùng 1.200 review như tầng 1, nên dùng cùng lưới với tầng 1: learning rate {1e-5, 2e-5, 3e-5} và số epoch {3, 4, 5}.

**Tokenizer**

Bản sentencepiece đã sửa ở Giai đoạn 6 (hàm huấn luyện tự chạy `self_check`).

In [ ]:
from src.classification.rq1 import VisobertRunner, grid_search, run_seeds, predictions_frame, cost_row, save_dev_config, find_checkpoints, verify_reload
from src.classification.flat_baseline.evaluate import argmax_metrics, summary_row, per_class_table, confusion_sum

FLAT_GRID = [{'lr': lr, 'epochs': ep} for lr in (1e-5, 2e-5, 3e-5) for ep in (3, 4, 5)]
vis = VisobertRunner(CFG['strategy_b_visobert'])
flat_grid, flat_best = grid_search(vis, FLAT_GRID, FLAT_TRAIN, FLAT_VAL, 4, FLAT_LABELS, argmax_metrics,
                                   select='macro_f1', seed=42, strategy='visobert_flat')
flat_grid.to_csv(os.path.join(OUT, 'flat_visobert_grid.csv'), index=False)
print(flat_grid[['config', 'macro_f1', 'f1_BUG', 'f1_FEATURE', 'f1_UX', 'f1_OTHER', 'train_seconds']].round(4).to_string(index=False))
print('Cấu hình thắng:', flat_best)

## Baseline: ba seed

In [ ]:
flat_runs = run_seeds(vis, flat_best, SEEDS, FLAT_TRAIN, FLAT_VAL, 4, FLAT_LABELS, strategy='visobert_flat',
                      save_dir=os.path.join(CKPT, 'visobert'))
predictions_frame(FLAT_VAL, flat_runs, FLAT_LABELS, 'visobert', 'flat').to_csv(
    os.path.join(OUT, 'flat_visobert_val_predictions.csv'), index=False)

## Kết quả baseline trên Validation

**Bảng chính**

Macro-F1 4 lớp và F1 từng lớp, trung bình và độ lệch chuẩn qua ba seed.

**Precision và recall**

Bảng thứ hai tách precision và recall của từng lớp.

**Ma trận nhầm lẫn**

Trung bình mỗi lần chạy (cộng ba seed rồi chia ba), để thấy baseline nhầm giữa các lớp actionable với nhau hay nhầm với OTHER.

In [ ]:
y4 = FLAT_VAL['y'].to_numpy()
flat_row = summary_row(y4, flat_runs, 'Baseline phẳng ViSoBERT', flat_best, 'Validation')
flat_pc = per_class_table(y4, flat_runs, 'Baseline phẳng ViSoBERT', 'Validation')
flat_cm = confusion_sum(y4, flat_runs) / len(flat_runs)
flat_cost = pd.DataFrame([cost_row(flat_runs, 'Baseline phẳng ViSoBERT')])
pd.DataFrame([flat_row]).to_csv(os.path.join(OUT, 'flat_validation_results.csv'), index=False)
flat_pc.to_csv(os.path.join(OUT, 'flat_validation_per_class.csv'), index=False)
flat_cm.to_csv(os.path.join(OUT, 'flat_visobert_confusion_mean.csv'))
flat_cost.to_csv(os.path.join(OUT, 'flat_costs.csv'), index=False)
print('Baseline phẳng 4 lớp trên Validation (%, trung bình ± độ lệch chuẩn, 3 seed):')
print(pd.DataFrame([flat_row]).drop(columns=['tập']).to_string(index=False))
print('\nPrecision / recall / F1 từng lớp:'); print(flat_pc.drop(columns=['tập']).to_string(index=False))
print('\nMa trận nhầm lẫn (trung bình mỗi lần chạy):'); print(flat_cm.round(1).to_string())
print('\nChi phí:'); print(flat_cost.round(3).to_string(index=False))
save_dev_config(os.path.join(OUT, 'flat_dev_config.yaml'),
                {'flat': {'visobert': {'overrides': flat_best, 'select': 'macro_f1 (4 lớp) trên Validation, argmax',
                                       'checkpoints': [os.path.relpath(r['checkpoint'], OUT).replace(os.sep, '/') for r in flat_runs]}}},
                'Baseline phẳng 4 lớp (reference only) — cấu hình thắng trên Validation. DEV, chưa frozen.\n'
                'Checkpoint 3 seed để GĐ10 so cascade với baseline trên Hold-out.')

## So sơ bộ: cascade ViSoBERT và baseline phẳng (Validation)

**Cách ghép cascade**

Tầng 1 dùng xác suất ACTIONABLE đã lưu ở Giai đoạn 6 với ngưỡng 0,11 (amendment v2.0): review nào tầng 1 gọi NOT_ACTIONABLE thì nhận
nhãn OTHER; review nào tầng 1 gọi ACTIONABLE thì nhận nhãn của tầng 2.

**Ghép theo cặp seed**

Tầng 1 seed 42 đi với tầng 2 seed 42, và tương tự cho 1337, 2024.

**Kiểm tra checkpoint nạp lại**

Mô hình tầng 2 nạp lại phải cho đúng kết quả của Giai đoạn 7 trên phần gold-actionable. Điều kiện chính là nhãn dự đoán khớp 100%, vì
nhãn mới là thứ quyết định mọi chỉ số. Xác suất được phép lệch một chút (dưới 0,01): chạy trên GPU hoặc phiên bản thư viện khác làm các
phép tính dấu phẩy động khác đi rất nhỏ, còn một checkpoint hỏng thật sẽ lệch xa hơn nhiều; cả tỉ lệ nhãn khớp lẫn mức chênh xác suất
đều được in ra.

**Thêm một nguồn lạc quan cho cascade**

Hai cách đều được chọn cấu hình trên cùng Validation, nhưng cascade còn thêm một ngưỡng tầng 1 cũng chọn trên Validation, nên con số
của cascade ở đây có thêm một nguồn lạc quan.

**Chi phí của cascade**

Chi phí của cascade là tổng hai tầng: huấn luyện cả hai mô hình, và lúc dự đoán thì tầng 2 chỉ chạy trên phần review được tầng 1 giữ
lại.

In [ ]:
from src.classification import evaluate as E
DEV = os.path.join(REPO_DIR, '03_configs', 'dev')
RES = os.path.join(REPO_DIR, '05_experiments', 'rq1_classification')
THR = yaml.safe_load(open(os.path.join(DEV, 'tier1_dev_config.yaml'), encoding='utf-8'))['tier1']['visobert']['threshold']
t1 = pd.read_csv(os.path.join(RES, 'tier1_visobert_val_predictions.csv'))
t2_saved = pd.read_csv(os.path.join(RES, 'tier2_visobert_val_predictions.csv'))
t2_paths = find_checkpoints('visobert', SEEDS, contains='rq1_tier2')
ids, texts = FLAT_VAL['review_id'].to_numpy(), FLAT_VAL['text'].astype(str).tolist()
casc_runs = []
for s, path in zip(SEEDS, t2_paths):
    runner = VisobertRunner(CFG['strategy_b_visobert'])
    model = runner.load(path)
    p2, _ = runner.predict_proba(model, texts)
    saved = t2_saved[t2_saved['seed'] == s].set_index('review_id')[['p_BUG', 'p_FEATURE', 'p_UX']]
    on_gold = np.isin(ids, saved.index)
    chk = verify_reload(p2[on_gold], saved.loc[ids[on_gold]].to_numpy(), name=f'tầng 2 seed {s}')   # nhãn 100% + |Δp| < 1e-2
    act = t1[t1['seed'] == s].set_index('review_id').loc[ids, 'p_ACTIONABLE'].to_numpy() >= THR
    casc_runs.append({'seed': s, 'pred': E.cascade_labels(act, p2.argmax(1)), 'tier1_actionable': act})
    print(f'seed {s}: tầng 1 giữ {act.sum()} / {len(act)} review | {chk["message"]}')
    del model; gc.collect(); torch.cuda.empty_cache()

pd.concat([pd.DataFrame({'review_id': ids, 'gold_y': y4, 'seed': r['seed'], 'tier1_actionable': r['tier1_actionable'],
                         'pred': r['pred']}) for r in casc_runs]).to_csv(os.path.join(OUT, 'cascade_visobert_val_predictions.csv'), index=False)
casc_row = summary_row(y4, casc_runs, 'Cascade ViSoBERT', f'tầng 1 ngưỡng {THR} + tầng 2', 'Validation')
cmp = pd.DataFrame([flat_row, casc_row])
fm, cm_ = E.runs_metrics(y4, flat_runs, FLAT_LABELS), E.runs_metrics(y4, casc_runs, FLAT_LABELS)
keys = ['macro_f1'] + [f'f1_{c}' for c in FLAT_LABELS]
delta = {'strategy': 'Chênh (cascade − baseline), điểm %', **{k: f'{(cm_[k].mean() - fm[k].mean()) * 100:+.1f}' for k in keys}}
cmp = pd.concat([cmp, pd.DataFrame([delta])])
cmp.to_csv(os.path.join(OUT, 'cascade_vs_flat_validation.csv'), index=False)
print('\nMacro-F1 4 lớp trên Validation (%, 3 seed):')
print(cmp[['strategy'] + keys].to_string(index=False))
print('\nChênh Macro-F1 theo từng cặp seed (cascade − baseline):',
      {s: round((c - f) * 100, 1) for s, c, f in zip(SEEDS, cm_['macro_f1'], fm['macro_f1'])})
print('\nMa trận nhầm lẫn cascade (trung bình mỗi lần chạy):'); print((confusion_sum(y4, casc_runs) / len(casc_runs)).round(1).to_string())

c1 = pd.read_csv(os.path.join(RES, 'tier1_costs.csv')).set_index('strategy').loc['ViSoBERT']
c2 = pd.read_csv(os.path.join(RES, 'tier2_costs.csv')).set_index('strategy').loc['ViSoBERT']
act_rate = float(np.mean([r['tier1_actionable'].mean() for r in casc_runs]))
cost_cmp = pd.DataFrame([
    {'cách': 'Baseline phẳng', 'mô hình cần train': 1, 'giây train / lần': flat_cost['train_seconds_mean'].iloc[0],
     'giây dự đoán / 1000 review': flat_cost['infer_seconds_per_1000_mean'].iloc[0], 'GPU tối đa (GB)': flat_cost['peak_gpu_gb_mean'].iloc[0]},
    {'cách': 'Cascade (tầng 1 + tầng 2)', 'mô hình cần train': 2, 'giây train / lần': c1['train_seconds_mean'] + c2['train_seconds_mean'],
     'giây dự đoán / 1000 review': c1['infer_seconds_per_1000_mean'] + act_rate * c2['infer_seconds_per_1000_mean'],
     'GPU tối đa (GB)': max(c1['peak_gpu_gb_mean'], c2['peak_gpu_gb_mean'])}])
cost_cmp.to_csv(os.path.join(OUT, 'cascade_vs_flat_costs.csv'), index=False)
print(f'\nChi phí (tầng 2 chạy trên {act_rate:.0%} review được tầng 1 giữ lại):'); print(cost_cmp.round(2).to_string(index=False))
print('\nĐã lưu:', sorted(f for f in os.listdir(OUT) if f != 'checkpoints'))

## Nhận xét

**Cấu hình thắng của baseline**

Learning rate 2e-5 với 5 epoch (Macro-F1 0,686 ở lần dò với seed 42). Lưới của baseline khá phẳng: năm cấu hình đứng đầu chỉ cách nhau
0,4 điểm, nên việc cấu hình thắng nằm ở mép số epoch không đáng lo như ở tầng 2.

**Kết quả baseline**

Chạy lại ba seed, baseline đạt Macro-F1 4 lớp 69,1 ± 2,0, với F1 BUG 50,7, FEATURE 58,9, UX 72,7 và OTHER 94,0. Precision của các lớp
actionable khá cao (BUG 59,2, FEATURE 79,2, UX 80,0) nhưng recall thấp (44,4; 47,0; 66,7).

**Lỗi chủ yếu: đẩy review actionable về OTHER**

Mỗi lần chạy, trung bình 12,7 trên 27 review BUG, 9,0 trên 22 review FEATURE và 5,3 trên 22 review UX bị gọi là OTHER, trong khi giữa
ba lớp actionable thì baseline ít nhầm.

**Macro-F1 4 lớp: cascade và baseline hòa nhau**

So với cascade ViSoBERT (tầng 1 ngưỡng 0,11 rồi tầng 2), Macro-F1 4 lớp gần như bằng nhau: 68,8 ± 0,8 so với 69,1 ± 2,0, chênh −0,3
điểm, và dấu của mức chênh đổi theo cặp seed (+1,0; −3,2; +1,4). Theo đúng thước đo Macro-F1 4 lớp, trên Validation chưa thấy cascade
hơn baseline.

**Kiểm tra checkpoint**

Trước khi ghép, mô hình tầng 2 của cả ba seed nạp lại từ Giai đoạn 7 đều cho nhãn khớp 100% trên phần gold-actionable.

**Hai điểm làm việc khác nhau**

Gộp ba lớp actionable lại, baseline chỉ giữ được 62,0% review actionable (precision 85,2), còn cascade giữ được 82,6% (precision 71,6);
trung bình mỗi lần chạy baseline gọi khoảng 52 review là actionable, cascade khoảng 82. Macro-F1 4 lớp chấm hai kiểu sai này gần như
ngang nhau nên điểm tổng không phân biệt được.

**Lợi thế thiết kế của cascade: ngưỡng điều chỉnh được**

Với mục tiêu của đề tài, tầng 1 được đặt ưu tiên recall (sàn 0,80), vì review actionable bị bỏ sót sẽ mất hẳn khỏi bước gom cụm và sinh
requirement, còn review OTHER lọt vào vẫn có thể bị lọc ở các bước sau. Baseline dùng argmax nên không có chỗ để chỉnh điểm làm việc
này, còn cascade có ngưỡng tầng 1; đây là lợi thế về thiết kế, không phải về điểm số.

**Ngưỡng cho baseline chưa được thử**

Có thể thêm cho baseline một ngưỡng trên xác suất OTHER để tăng recall, nhưng việc này không được làm, vì đó là thêm một lần tune trên
Validation trong khi protocol chỉ đặt baseline làm mốc tham chiếu.

**Giá của recall cao: nhiễu dồn vào FEATURE**

Mỗi lần chạy, cascade đưa trung bình 23,7 review OTHER vào nhóm actionable, so với 7,7 review của baseline, và tầng 2 gán phần lớn
chúng là FEATURE (11,7) và BUG (8,3). Đây là lý do F1 FEATURE của cascade thấp hơn baseline 2,6 điểm dù tầng 2 phân FEATURE khá tốt:
precision FEATURE bị kéo xuống bởi review OTHER lọt qua tầng 1.

**Phân lớp trong phần actionable được giữ lại**

Baseline phân đúng lớp 84,1%, cascade 78,4%, và cascade nhầm BUG/FEATURE nhiều hơn (5,3 so với 2,3 review mỗi lần chạy); một phần là vì
cascade giữ lại cả những review actionable khó mà baseline đã đẩy về OTHER.

**Chi phí gần như ngang nhau**

Baseline huấn luyện một mô hình trong khoảng 40,7 giây (5 epoch), cascade huấn luyện hai mô hình tổng khoảng 43,4 giây (24,9 giây tầng
1 và 18,4 giây tầng 2). Dự đoán 1.000 review mất 1,38 giây với baseline và khoảng 1,85 giây với cascade, vì tầng 2 chỉ chạy trên khoảng
24% review được tầng 1 giữ lại; bộ nhớ GPU tối đa đều khoảng 2,5 GB. Khác biệt chính nằm ở độ phức tạp: cascade cần hai mô hình và một
ngưỡng.

**Mỗi cách lạc quan theo một kiểu**

Các con số ở đây đều lạc quan theo những cách khác nhau. Baseline được chọn cấu hình bằng chính Macro-F1 4 lớp trên Validation, tức
đúng thước đo đang so sánh, còn các tầng của cascade được chọn theo thước đo riêng của từng tầng và ngưỡng tầng 1 được chọn để đạt
recall chứ không để tối đa Macro-F1 4 lớp.

**Chưa kết luận**

Chưa kết luận cascade hay baseline tốt hơn; so sánh chính thức để ở Giai đoạn 10 trên Hold-out, bằng các mô hình đã lưu ở Giai đoạn 7
và ở notebook này.

**File kết quả và checkpoint**

Các file kết quả trong `/kaggle/working/rq1_flat/` (trừ thư mục `checkpoints/`) đã được tải về `05_experiments/rq1_classification/`, và
`flat_dev_config.yaml` đã được chép vào `03_configs/dev/`. Thư mục `checkpoints/` được giữ trong output của phiên bản notebook này để
gắn làm input cho Giai đoạn 10.